In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# 1. Загрузите данные из файла data-logistic.csv. Это двумерная выборка, целевая переменная на которой принимает значения -1 или 1.

In [3]:
data = pd.read_csv('data-logistic.csv', header=None)

y_true = data.iloc[:, 0].values.astype(float)
X_features = data.iloc[:, 1:3].values.astype(float)


# 2. Убедитесь, что выше выписаны правильные формулы для градиентного спуска.Обратите внимание, что мы используем полноценный градиентный спуск, а не его стохастический вариант!

In [4]:
def gradient_descent_logistic(X, y, C, step=0.1, max_iters=10000, tol=1e-5, w_init=None):
    """
    X: матрица признаков (n_samples, 2)
    y: вектор классов (-1, 1)
    C: коэффициент регуляризации (0 - без регуляризации)
    step: шаг градиентного спуска (k)
    max_iters: максимальное число итераций
    tol: критерий остановки (изменение весов)
    w_init: начальные веса (по умолчанию (0,0))
    """
    n_samples = len(y)
    weights = np.zeros(2) if w_init is None else w_init.copy()

    for iteration in range(1, max_iters + 1):
        # Вычисляем линейную комбинацию
        linear_pred = X @ weights

        # Сигмоида
        # Ограничиваем аргумент exp, чтобы избежать переполнения
        clipped = np.clip(-y * linear_pred, -700, 700)
        prob = 1.0 / (1.0 + np.exp(clipped))

        # Градиент функции потерь (без регуляризации)
        grad_main = (X.T @ (y * (1.0 - prob))) / n_samples

        # Добавляем регуляризацию (штраф L2)
        grad_reg = C * weights
        gradient = grad_main - grad_reg

        # Обновляем веса
        new_weights = weights + step * gradient

        # Проверка на сходимость
        if np.linalg.norm(new_weights - weights) <= tol:
            weights = new_weights
            return weights, iteration, True

        weights = new_weights

    return weights, max_iters, False

# 3. Реализуйте градиентный спуск для обычной и L2-регуляризованной (с коэффициентом регуляризации 10) логистической регрессии. Используйте длину шага k=0.1. В качестве начального приближения используйте вектор (0, 0).

# 4. Запустите градиентный спуск и доведите до сходимости (евклидово расстояние между векторами весов на соседних итерациях должно быть не больше 1e-5). Рекомендуется ограничить сверху число итераций десятью тысячами.

In [5]:
print("\n" + "="*50)
print("Обучение без регуляризации (C=0)")
weights_no_reg, iters_no_reg, converged_no = gradient_descent_logistic(
    X_features, y_true, C=0.0, step=0.1
)
print(f"  Итераций: {iters_no_reg}")
print(f"  Сошелся: {converged_no}")
print(f"  Веса: w1={weights_no_reg[0]:.6f}, w2={weights_no_reg[1]:.6f}")


Обучение без регуляризации (C=0)
  Итераций: 244
  Сошелся: True
  Веса: w1=0.287812, w2=0.091983


In [6]:
print("\n" + "="*50)
print("Обучение с регуляризацией (C=10)")
weights_reg, iters_reg, converged_reg = gradient_descent_logistic(
    X_features, y_true, C=10.0, step=0.1
)
print(f"  Итераций: {iters_reg}")
print(f"  Сошелся: {converged_reg}")
print(f"  Веса: w1={weights_reg[0]:.6f}, w2={weights_reg[1]:.6f}")


Обучение с регуляризацией (C=10)
  Итераций: 8
  Сошелся: True
  Веса: w1=0.028559, w2=0.024780


# 5. Какое значение принимает AUC-ROC на обучении без регуляризации и при ее использовании? Эти величины будут ответом на задание. В качестве ответа приведите два числа через пробел. Обратите внимание, что на вход функции roc_auc_score нужно подавать оценки вероятностей, подсчитанные обученным алгоритмом. Для этого воспользуйтесь сигмоидной функцией: a(x) = 1/(1 +exp(−w1x1 − w2x2)).

In [8]:
def predict_proba(weights, X):
    logits = X @ weights
    clipped = np.clip(-logits, -700, 700)  # для устойчивости
    return 1.0 / (1.0 + np.exp(clipped))

proba_no_reg = predict_proba(weights_no_reg, X_features)
proba_reg = predict_proba(weights_reg, X_features)

# AUC-ROC
auc_no_reg = roc_auc_score(y_true, proba_no_reg)
auc_reg = roc_auc_score(y_true, proba_reg)

print(f"AUC без регуляризации (C=0):  {auc_no_reg:.3f}")
print(f"AUC с регуляризацией (C=10): {auc_reg:.3f}")

AUC без регуляризации (C=0):  0.927
AUC с регуляризацией (C=10): 0.936


# 6. Попробуйте поменять длину шага. Будет ли сходиться алгоритм, если делать более длинные шаги? Как меняется число итераций при уменьшении длины шага?

In [12]:
for test_k in [1.0, 0.1, 0.01]:
    _, iters, conv = gradient_descent_logistic(X_features, y_true, C=0.0, step=test_k)
    print(f"k={test_k:4.2f} | Сходится: {'ДА' if conv else 'НЕТ'} | Итераций: {iters}")

print("При более длинных шагах алгоритм может расходиться.")
print("При уменьшении шага число итераций увеличивается.")

k=1.00 | Сходится: ДА | Итераций: 32
k=0.10 | Сходится: ДА | Итераций: 244
k=0.01 | Сходится: ДА | Итераций: 1479
При более длинных шагах алгоритм может расходиться.
При уменьшении шага число итераций увеличивается.


# 7. Попробуйте менять начальное приближение. Влияет ли оно на что-нибудь?

In [14]:
w_base, it_base, _ = gradient_descent_logistic(X_features, y_true, C=0.0, step=0.1, w_init=np.array([0.0, 0.0]))
w_alt, it_alt, _   = gradient_descent_logistic(X_features, y_true, C=0.0, step=0.1, w_init=np.array([10.0, -5.0]))

proba_base = predict_proba(w_base, X_features)
proba_alt = predict_proba(w_alt, X_features)

auc_base = roc_auc_score(y_true, proba_base)
auc_alt = roc_auc_score(y_true, proba_alt)

print(f"Старт (0,0)   → AUC: {auc_base:.3f}, итераций: {it_base}")
print(f"Старт (10,-5) → AUC: {auc_alt:.3f}, итераций: {it_alt}")
print(f"Разница финальных весов: {np.linalg.norm(w_base - w_alt):.2e}")

print("Начальное приближение НЕ влияет на точность (AUC одинаков),")
print("но влияет на скорость сходимости (число итераций).")

Старт (0,0)   → AUC: 0.927, итераций: 244
Старт (10,-5) → AUC: 0.927, итераций: 656
Разница финальных весов: 8.09e-04
Начальное приближение НЕ влияет на точность (AUC одинаков),
но влияет на скорость сходимости (число итераций).
